# Research Question 5

**Does the model know when it is wrong?**

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
from IPython.utils.capture import capture_output

In [3]:
# Import data from the base notebook

# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run r0_dataset_overiew_dashboard.ipynb

In [5]:
HTML_OUTPUT_FOLDER = "html_plots/r5"

In [6]:
results = []

for pathology in DATASET_PATHOLOGIES:

    label_idx = DATASET_LABEL_INDICES_MAP[pathology]

    y_true = XRAY_LABELS_LIST[:, label_idx]
    y_score = MODEL_SCORE_PREDICTIONS_CONVERTED[:, label_idx]

    y_pred = (y_score >= 0.7).astype(int)

    fp = (y_true == 0) & (y_pred == 1)

    fp_scores = y_score[fp]

    if len(fp_scores) == 0:
        continue

    results.append({
        "pathology": pathology,
        "n_false_positives": len(fp_scores),
        "mean_fp_score": fp_scores.mean(),
        "median_fp_score": np.median(fp_scores),
        "max_fp_score": fp_scores.max(),
        "fp_ge_0.9": np.mean(fp_scores >= 0.9),
        "fp_ge_0.95": np.mean(fp_scores >= 0.95)
    })

fp_results = pd.DataFrame(results)

fp_results.sort_values(
    "fp_ge_0.9",
    ascending=False
)

,pathology,n_false_positives,mean_fp_score,median_fp_score,max_fp_score,fp_ge_0.9,fp_ge_0.95
7,Lung Opacity,2020,0.856300,0.865689,0.992328,0.362871,0.131188
8,Effusion,2026,0.852543,0.859616,0.993640,0.338105,0.122409
1,Cardiomegaly,2681,0.844830,0.848395,0.996576,0.314808,0.131294
0,Atelectasis,3953,0.844834,0.848893,0.988118,0.260815,0.065773
3,Edema,2527,0.820718,0.820154,0.986026,0.180451,0.036803
6,Lung Lesion,1217,0.807711,0.791715,0.994605,0.151191,0.053410
2,Consolidation,2322,0.813248,0.808455,0.982610,0.133075,0.024978
4,Enlarged Cardiomediastinum,1812,0.801258,0.790553,0.985788,0.108720,0.023731
5,Fracture,1005,0.789615,0.775129,0.986304,0.085572,0.018905
9,Pneumonia,1383,0.784395,0.772613,0.972478,0.060738,0.009400


In [7]:
def prepare_pathology_data(pathology, threshold=THRESHOLD):

    label_idx = DATASET_LABEL_INDICES_MAP[pathology]


    y_true_raw = np.asarray(XRAY_LABELS_LIST[:, label_idx])
    y_score_raw = np.asarray(
        MODEL_SCORE_PREDICTIONS_CONVERTED[:, label_idx]
    )

    valid = np.isin(y_true_raw, [0, 1])

    y_true = y_true_raw[valid].astype(int)
    y_score = y_score_raw[valid]

    y_pred = (y_score >= threshold).astype(int)

    # Classification categories
    tp = (y_true == 1) & (y_pred == 1)
    tn = (y_true == 0) & (y_pred == 0)
    fp = (y_true == 0) & (y_pred == 1)
    fn = (y_true == 1) & (y_pred == 0)

    # Confidence in the predicted class
    prediction_confidence = np.where(
        y_pred == 1,
        y_score,
        1 - y_score
    )

    df = pd.DataFrame({
        "ground_truth": y_true,
        "prediction": y_pred,
        "score": y_score,
        "prediction_confidence": prediction_confidence,
        "outcome": np.select(
            [tp, tn, fp, fn],
            ["True Positive", "True Negative",
             "False Positive", "False Negative"], default=None
        ),
        "age": np.asarray(XRAY_AGE_LIST)[valid],
        "sex": np.asarray(XRAY_GENDER_LIST)[valid],
        "view": np.asarray(XRAY_VIEW_LIST)[valid]
    })

    return df

In [8]:
def show_prediction_confidence_vs_probability(pathology, threshold = THRESHOLD, output_folder=None):

    df = prepare_pathology_data(pathology, threshold=threshold)

    df.head()
    fig = px.histogram(
        df,
        x="prediction_confidence",
        color="outcome",
        barmode="overlay",
        histnorm="probability density",
        nbins=40,
        opacity=0.65,
        color_discrete_map={
            "True Positive": "#009E73",
            "True Negative": "#0072B2",
            "False Positive": "#D55E00",
            "False Negative": "#CC79A7"
        },
        hover_data=["score", "age", "sex", "view"],
        title=f"Does the Model Know When It Is Wrong? – {pathology}"
    )

    fig.add_vline(
        x=0.9,
        line_dash="dash",
        line_color="black",
        annotation_text="High confidence (0.9)"
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_title="Confidence in predicted class",
        yaxis_title="Probability density",
        legend_title="Prediction outcome",
        hovermode="x unified",
        width=1000,
        height=600
    )

    if output_folder is not None:
        fig.write_html(output_folder + f'/prediction_prob_{threshold}_{pathology}.html',
                       include_plotlyjs="cdn",
                       full_html=True)

    fig.show()

In [9]:
for pathology in DATASET_PATHOLOGIES:
    show_prediction_confidence_vs_probability(pathology, output_folder=HTML_OUTPUT_FOLDER)

### Observation

Finding: The model is sometimes highly confident when it is wrong.

A possible interpretation: The model does not reliably distinguish correct positive predictions form incorrect positive predictions.

Finding: The Pneumothorax plot is especially unusual: Both true positives and false positives have a strong concentration just around the treshold.

The model seems to produce many borderline positive predictions and barly predicts any positive cases anyways.

## 2 

In [19]:
def show_confidence_dist(pathology, output_folder=None):
    df = prepare_pathology_data(pathology)
    fp_df = df[df["outcome"] == "False Positive"].copy()

    fig = px.histogram(
        fp_df,
        x="score",
        nbins=40,
        color_discrete_sequence=["#E74C3C"],
        marginal="box",
        title=f"Confidence Distribution of False Positives – {pathology}",
        hover_data=["age", "sex", "view"]
    )

    fig.add_vline(
        x=0.7,
        line_dash="dash",
        line_color="black",
        annotation_text="Decision threshold (0.7)"
    )

    fig.add_vline(
        x=0.9,
        line_dash="dot",
        line_color="blue",
        annotation_text="High confidence (0.9)"
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_title="Model score",
        yaxis_title="Number of false positives",
        width=1000,
        height=600
    )

    if output_folder is not None:
        fig.write_html(output_folder + f'/conf_dist_{pathology}.html',
                       include_plotlyjs="cdn",
                       full_html=True)

    fig.show()

In [20]:
for pathology in DATASET_PATHOLOGIES:
    show_confidence_dist(pathology, output_folder=HTML_OUTPUT_FOLDER)

### Observation

Atelectasis, Cardiomegaly, Lung Opacity and Effusion exhibit broad false-positive distributions, with substantial portions extending above 0.9.

Fracture, Pneumonia and Enlarged Cardiome­diastinum show stronger concentrations of false positives around the decision threshold of 0.7–0.8.